# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Himanshu-rai-prog/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Baseline rule:

Review a page when it has not been updated for at least 181 days and still has at least 500 impressions in the last 90 days.

Score = impressions_90d when both conditions are true.

Reason code = stale_but_visible

Action = review_refresh

The rule is intentionally transparent and uses no fitted weights.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/Himanshu-rai-prog/flyrank-ml-internship/refs/heads/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [4]:
# Signal 1: Staleness bucket check

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30", "31-90", "91-180", "181+"]
)

print("STALENESS SIGNAL")
print(df["staleness_bucket"].value_counts().sort_index())

STALENESS SIGNAL
staleness_bucket
0-30      20480
31-90       175
91-180     9171
181+        174
Name: count, dtype: int64


In [6]:
# Staleness signal audit
audit = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean"),
          avg_position=("avg_position", "mean"),
          avg_impressions=("impressions_90d", "mean")
      )
      .reset_index()
)

print(audit.to_string(index=False))

staleness_bucket     n  avg_ctr  avg_position  avg_impressions
            0-30 20480 0.609021     15.685166      4199.614062
           31-90   175 0.117543     16.538286      6506.748571
          91-180  9171 0.238367     17.901461      7486.665140
            181+   174 3.693276     11.325862      1172.448276


Signal 1 — Staleness: MIXED

The 181+ days bucket has only 174 rows and lower average impressions than the 0–30 and 31–90 day buckets. This means staleness alone does not consistently indicate higher visibility, so the signal is mixed. I still use staleness as the main refresh-review condition because it is directly related to the refresh use case.

Signal 2 — CTR vs position: CONFIRMED

Average CTR decreases as average position gets worse: the 1–3 bucket has the highest average CTR, while the 21+ bucket has the lowest. This supports a relationship between position and CTR.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------
# 1. SIGNAL CHECKS
# -----------------------------

# Signal 1: Staleness
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30", "31-90", "91-180", "181+"]
)

stale_audit = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_impressions=("impressions_90d", "mean"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

print("=== SIGNAL 1: STALENESS ===")
print(stale_audit.to_string(index=False))

# Signal 2: CTR vs Position
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)

position_audit = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

print("\n=== SIGNAL 2: CTR VS POSITION ===")
print(position_audit.to_string(index=False))


# -----------------------------
# 2. TRANSPARENT BASELINE RULE
# -----------------------------

# Rule:
# Review pages that are old AND still receive meaningful impressions.

df["stale"] = (df["days_since_last_update"] >= 181).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)

df["score"] = df["stale"] * df["visible"] * df["impressions_90d"]

df["reason_code"] = np.where(
    (df["stale"] == 1) & (df["visible"] == 1),
    "stale_but_visible",
    "not_selected"
)

df["action"] = np.where(
    df["reason_code"] == "stale_but_visible",
    "review_refresh",
    "no_action"
)

# Rank highest score first
queue = (
    df.sort_values(["score", "impressions_90d"], ascending=False)
      .reset_index(drop=True)
)

# -----------------------------
# 3. WRITE RANKED QUEUE
# -----------------------------

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "baseline_action_score.csv"

queue[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position"
    ]
].to_csv(output_file, index=False)

print("\n=== QUEUE CREATED ===")
print("Rows:", len(queue))
print("CSV:", output_file)


# -----------------------------
# 4. TOP-10 REVIEW
# -----------------------------

top10 = queue.head(10)

print("\n=== TOP-10 REVIEW ===")

for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(
        f"{i}. Action: {row['action']} | "
        f"Why: {row['reason_code']} | "
        f"Score: {row['score']:.0f} | "
        f"What would make it wrong: impressions may be low-quality, "
        f"the page may already be intentionally evergreen, or the refresh "
        f"may not be needed."
    )

=== SIGNAL 1: STALENESS ===
staleness_bucket     n  avg_impressions  avg_ctr
            0-30 20480      4199.614062 0.609021
           31-90   175      6506.748571 0.117543
          91-180  9171      7486.665140 0.238367
            181+   174      1172.448276 3.693276

=== SIGNAL 2: CTR VS POSITION ===
position_bucket     n  avg_ctr
            1-3  1141 2.714303
           4-10 11842 0.651045
          11-20  7273 0.323443
            21+  8539 0.211333

=== QUEUE CREATED ===
Rows: 30000
CSV: work/outputs/baseline_action_score.csv

=== TOP-10 REVIEW ===
1. Action: review_refresh | Why: stale_but_visible | Score: 61678 | What would make it wrong: impressions may be low-quality, the page may already be intentionally evergreen, or the refresh may not be needed.
2. Action: review_refresh | Why: stale_but_visible | Score: 59472 | What would make it wrong: impressions may be low-quality, the page may already be intentionally evergreen, or the refresh may not be needed.
3. Action: review

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for i, col in enumerate(df.columns, 1):
    print(i, ":", col)


1 : content_id
2 : client_id
3 : search_volume
4 : competition
5 : competition_level
6 : cpc
7 : content_type
8 : main_intent
9 : word_count
10 : char_count
11 : provider_used
12 : model_used
13 : impressions_90d
14 : clicks_90d
15 : pageviews_90d
16 : sessions_90d
17 : users_90d
18 : engaged_sessions_90d
19 : ai_sessions_90d
20 : scroll_events_90d
21 : days_with_impressions
22 : days_with_sessions
23 : impressions_last_30d
24 : clicks_last_30d
25 : sessions_last_30d
26 : impressions_prev_30d
27 : clicks_prev_30d
28 : sessions_prev_30d
29 : content_age_days
30 : age_tier
31 : age_tier_order
32 : days_since_last_update
33 : freshness_tier
34 : word_count_tier
35 : char_count_tier
36 : ctr
37 : avg_position
38 : engagement_rate
39 : scroll_rate
40 : ai_traffic_pct
41 : impression_tier
42 : position_tier
43 : trend_direction
44 : trend_pct


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)

print("=== TOP-10 REVIEW ===")

for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(
        f"{i}. content_id={row['content_id']} | "
        f"Action={row['action']} | "
        f"Why={row['reason_code']} | "
        f"Score={row['score']:.0f} | "
        f"What would make it wrong: the impressions may not represent useful traffic, "
        f"the page may already be intentionally evergreen, or a refresh may not be needed."
    )

=== TOP-10 REVIEW ===
1. content_id=content_cf56e2e2e282 | Action=review_refresh | Why=stale_but_visible | Score=61678 | What would make it wrong: the impressions may not represent useful traffic, the page may already be intentionally evergreen, or a refresh may not be needed.
2. content_id=content_7368877ea310 | Action=review_refresh | Why=stale_but_visible | Score=59472 | What would make it wrong: the impressions may not represent useful traffic, the page may already be intentionally evergreen, or a refresh may not be needed.
3. content_id=content_1bfaa38ff26c | Action=review_refresh | Why=stale_but_visible | Score=25715 | What would make it wrong: the impressions may not represent useful traffic, the page may already be intentionally evergreen, or a refresh may not be needed.
4. content_id=content_0a91db491d14 | Action=review_refresh | Why=stale_but_visible | Score=13299 | What would make it wrong: the impressions may not represent useful traffic, the page may already be intentional

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check
for col in ["trend_direction", "trend_pct"]:
    assert col not in [
        "score",
        "stale",
        "visible"
    ]

print("Leakage check: trend_direction and trend_pct were not used in the score.")
print("Future-window inputs were not used.")
print("Weak-pick note: a high-impression stale page can still be a weak pick if the traffic is low-quality or the page is intentionally evergreen.")


Leakage check: trend_direction and trend_pct were not used in the score.
Future-window inputs were not used.
Weak-pick note: a high-impression stale page can still be a weak pick if the traffic is low-quality or the page is intentionally evergreen.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.